# LNP-MFGO Phase 2C — Adversarial Domain Adaptation, Multi-Task Learning & Ensemble

## What Phases 1–2B Established

| Phase | Key Finding |
|-------|-------------|
| 1 | Random-split R² = 0.44, but cross-study R² = −5 to −10 |
| 2A | Z-normalisation recovers R² ≈ 0; batch effects account for >99% of error |
| 2B | GNN matches descriptors (R² ≈ 0) but captures chemically meaningful attention |

**The bottleneck is not representation — it is domain shift between laboratories.**

## Phase 2C Strategy

We attack the domain shift problem with three synergistic interventions:

**1. Adversarial Domain Adaptation (Gradient Reversal)**  
Force the encoder to produce representations from which a discriminator *cannot* identify  
the source study. If the model can't tell which lab produced a sample, its predictions  
must rely on genuine molecular structure — not lab-specific artifacts.

**2. Multi-Task Learning**  
Predict all 4 physicochemical targets (particle size, PDI, EE%, zeta potential) + bioactivity  
simultaneously with a shared encoder. This acts as regularisation: the encoder must learn  
features useful for *multiple* properties, reducing overfitting to any single target.

**3. Ensemble: GNN + Descriptor Model**  
Phase 2B showed GNN wins some folds, descriptors win others. A learned ensemble combines  
their complementary strengths.

---

## Section 0: Environment Setup

In [ ]:
# ============================================================
# 0.1 — Imports & Configuration
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, json, copy
from pathlib import Path
from collections import defaultdict

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr, pearsonr
import xgboost as xgb
import lightgbm as lgb

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Function

from rdkit import Chem

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print(f'✅ CUDA: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  CPU mode')

PROJECT_DIR = Path('.')
OUTPUT_DIR = PROJECT_DIR / 'outputs'; OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR = OUTPUT_DIR / 'figures'; FIGURE_DIR.mkdir(exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 11,
                      'font.family': 'sans-serif', 'axes.grid': True, 'grid.alpha': 0.3})
COLORS = {'primary': '#2563EB', 'secondary': '#DC2626', 'tertiary': '#059669',
          'quaternary': '#D97706', 'purple': '#7C3AED'}
print('Setup complete ✅')

---
## Section 1: Data Preparation (Self-Contained)

We rebuild everything from the CSV — molecular graphs, feature tiers, Z-normalised  
targets, and study encodings. This makes Phase 2C fully standalone.

In [ ]:
# ============================================================
# 1.1 — Load & Prepare All Data
# ============================================================

df_raw = pd.read_csv(PROJECT_DIR / 'LNP_Atlas_Bioactivity_Reextracted.csv',
                      encoding='latin-1', low_memory=False)
print(f'Loaded: {df_raw.shape}')

# ---------- Z-Normalisation for ALL targets ----------
TARGETS = {
    'ps': 'particle_size_nm_std_num',
    'pdi': 'pdi_std_num',
    'ee': 'encapsulation_efficiency_percent_std_num',
    'zeta': 'zeta_potential_mv_std_num',
}

for tname, tcol in TARGETS.items():
    mask = df_raw[tcol].notna()
    gstats = df_raw[mask].groupby('paper_doi')[tcol].agg(['mean','std'])
    g_mean, g_std = df_raw[mask][tcol].mean(), df_raw[mask][tcol].std()
    gstats['std'] = gstats['std'].fillna(g_std)
    gstats.loc[gstats['std']==0, 'std'] = g_std
    df_raw[f'{tname}_smean'] = df_raw['paper_doi'].map(gstats['mean']).fillna(g_mean)
    df_raw[f'{tname}_sstd'] = df_raw['paper_doi'].map(gstats['std']).fillna(g_std)
    df_raw[f'{tname}_znorm'] = (df_raw[tcol] - df_raw[f'{tname}_smean']) / df_raw[f'{tname}_sstd']

# ---------- Bioactivity ordinal target ----------
df_raw['bio_ordinal'] = df_raw['Extracted_Bio_Ordinal_Class']

# ---------- Study encoding ----------
study_le = LabelEncoder()
df_raw['study_id'] = study_le.fit_transform(df_raw['paper_doi'].fillna('UNK'))
N_STUDIES = df_raw['study_id'].nunique()

# ---------- Molecular graphs ----------
ATOM_LIST = [6,7,8,9,15,16,17,35,53,0]
HYB_LIST = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
            Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2]
ATOM_DIM = 33
BOND_DIM = 6

def one_hot(val, allowed):
    enc = [0]*(len(allowed)+1)
    if val in allowed: enc[allowed.index(val)] = 1
    else: enc[-1] = 1
    return enc

def atom_feats(a):
    return (one_hot(a.GetAtomicNum(), ATOM_LIST) + one_hot(a.GetDegree(), [0,1,2,3,4,5])
            + one_hot(a.GetFormalCharge(), [-1,0,1,2]) + one_hot(a.GetNumRadicalElectrons(), [0,1])
            + one_hot(a.GetHybridization(), HYB_LIST) + [a.GetIsAromatic()])

def bond_feats(b):
    bt = b.GetBondType()
    return [float(x) for x in [bt==Chem.rdchem.BondType.SINGLE, bt==Chem.rdchem.BondType.DOUBLE,
            bt==Chem.rdchem.BondType.TRIPLE, bt==Chem.rdchem.BondType.AROMATIC,
            b.GetIsConjugated(), b.IsInRing()]]

def smiles_to_graph(smi):
    if pd.isna(smi) or not isinstance(smi, str) or len(smi) < 3: return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.FloatTensor([atom_feats(a) for a in mol.GetAtoms()])
    ei, ef = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = bond_feats(b)
        ei.extend([[i,j],[j,i]]); ef.extend([bf, bf])
    if not ei:
        return {'x': x, 'edge_index': torch.LongTensor([[0],[0]]),
                'edge_attr': torch.zeros(1, BOND_DIM), 'num_atoms': x.shape[0]}
    return {'x': x, 'edge_index': torch.LongTensor(ei).t().contiguous(),
            'edge_attr': torch.FloatTensor(ef), 'num_atoms': x.shape[0]}

LIPID_SMILES = ['ionizable_lipid_smiles','helper_lipid_smiles',
                'sterol_lipid_smiles','peg_lipid_smiles']
LIPID_NAMES = ['ionizable','helper','sterol','peg']

cache = {}
def get_g(s):
    if s not in cache: cache[s] = smiles_to_graph(s)
    return cache[s]

print('Building molecular graphs...')
all_graphs = {n: [] for n in LIPID_NAMES}
valid_idx = []
for idx, row in df_raw.iterrows():
    ok = True; rg = {}
    for ln, sc in zip(LIPID_NAMES, LIPID_SMILES):
        g = get_g(row[sc]) if pd.notna(row[sc]) else None
        if g is None: ok = False; break
        rg[ln] = g
    if ok:
        for n in LIPID_NAMES: all_graphs[n].append(rg[n])
        valid_idx.append(idx)

df = df_raw.loc[valid_idx].reset_index(drop=True)
print(f'Valid formulations: {len(df)} (all 4 SMILES parsed)')

# ---------- Descriptor features (Tier 1 rebuild) ----------
mordred_cols = [c for c in df.columns if c.startswith('mordred_')]
mdf = df[mordred_cols]
zero_var = mdf.columns[mdf.var() == 0].tolist()
mdf = mdf.drop(columns=zero_var)
nzv = [c for c in mdf.columns if mdf[c].value_counts(normalize=True).iloc[0] > 0.95]
mdf = mdf.drop(columns=nzv)
print(f'Computing correlation matrix for pruning...')
corr = mdf.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
hc = [c for c in upper.columns if any(upper[c] > 0.95)]
mordred_pruned = mdf.drop(columns=hc).columns.tolist()

rdkit_cols = ['MW','LogP','TPSA','HBD','HBA','RotBonds','HeavyAtoms',
              'Rings','AromaticRings','FractionCSP3','FormalCharge','NumAmines','NumAmides']
tab_cols_form = ['Ratio_Ionizable','Ratio_Helper','Ratio_Sterol','Ratio_PEG',
                 'Process_FlowRate','Process_Ratio_AqOrg','Process_Is_Microfluidic','Process_pH']
for c in ['helper_lipid','sterol_lipid','peg_lipid']:
    le = LabelEncoder()
    df[f'{c}_enc'] = le.fit_transform(df[c].fillna('UNK').astype(str))
lip_enc = ['helper_lipid_enc','sterol_lipid_enc','peg_lipid_enc']

TIER1 = rdkit_cols + mordred_pruned + tab_cols_form + lip_enc
TABULAR_GNN = tab_cols_form  # for GNN branch (ratios+process only)

print(f'Tier 1 descriptor features: {len(TIER1)}')
print(f'GNN tabular features: {len(TABULAR_GNN)}')
print(f'Studies: {N_STUDIES}')
print('Data preparation complete ✅')

---
## Section 2: Adversarial Domain Adaptation Architecture

The key innovation is a **Gradient Reversal Layer (GRL)**, introduced by Ganin et al. (2016).  
During forward pass, GRL is an identity function. During backward pass, it **negates the gradient**.  
This forces the encoder to produce features that MAXIMISE study-discrimination loss —  
i.e., features from which the study discriminator cannot determine the source laboratory.

```
                          ┌──────────────┐
  Molecular Graphs ──→ GIN Encoder ──→ │ Formulation  │──→ Property Predictor (MSE loss)
  Tabular Features ──→ MLP Encoder ──→ │ Embedding    │
                          └──────┬───────┘
                                 │
                          Gradient Reversal Layer (λ)
                                 │
                          Study Discriminator (CE loss)
```

The total loss is: `L = L_prediction − λ · L_study_discrimination`

λ starts at 0 and gradually increases (schedule from Ganin et al.), allowing the  
encoder to first learn useful features before the adversarial pressure kicks in.

In [ ]:
# ============================================================
# 2.1 — Gradient Reversal Layer
# ============================================================

class GradientReversalFunction(Function):
    """Gradient Reversal Layer: identity forward, negate gradient backward."""
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class GradientReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)


def grl_lambda_schedule(epoch, max_epochs, gamma=10.0):
    """Ganin et al. schedule: λ ramps from 0 to 1 over training."""
    p = epoch / max_epochs
    return 2.0 / (1.0 + np.exp(-gamma * p)) - 1.0

# Plot schedule
epochs_test = np.arange(150)
lambdas = [grl_lambda_schedule(e, 150) for e in epochs_test]
plt.figure(figsize=(8, 3))
plt.plot(epochs_test, lambdas, color=COLORS['primary'], linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('λ (GRL strength)')
plt.title('Gradient Reversal Schedule', fontweight='bold')
plt.tight_layout(); plt.show()
print('Gradient Reversal Layer ready ✅')

In [ ]:
# ============================================================
# 2.2 — GIN Encoder (reused from Phase 2B)
# ============================================================

class GINLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.eps = nn.Parameter(torch.zeros(1))
        self.mlp = nn.Sequential(nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim),
                                  nn.ReLU(), nn.Linear(out_dim, out_dim), nn.ReLU())
    def forward(self, x, edge_index, bna):
        row, col = edge_index
        agg = torch.zeros_like(x); agg.index_add_(0, col, x[row])
        return self.mlp((1 + self.eps) * x + agg)

class GINEncoder(nn.Module):
    def __init__(self, atom_dim=ATOM_DIM, hidden=128, out=128, layers=3, drop=0.2):
        super().__init__()
        self.emb = nn.Linear(atom_dim, hidden)
        self.convs = nn.ModuleList([GINLayer(hidden, hidden) for _ in range(layers)])
        self.drop = nn.Dropout(drop)
        self.proj = nn.Linear(hidden, out)
    def forward(self, x, ei, bna):
        h = self.emb(x)
        for conv in self.convs: h = self.drop(conv(h, ei, bna))
        gs = []; off = 0
        for n in bna: gs.append(h[off:off+n].mean(0)); off += n
        return self.proj(torch.stack(gs))

print('GIN Encoder ready ✅')

In [ ]:
# ============================================================
# 2.3 — Full Adversarial Multi-Task Model
# ============================================================

class CrossComponentAttention(nn.Module):
    def __init__(self, dim=128, heads=4, drop=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*2), nn.ReLU(), nn.Dropout(drop), nn.Linear(dim*2, dim))
        self.norm2 = nn.LayerNorm(dim)
    def forward(self, x):
        a, w = self.attn(x, x, x)
        x = self.norm1(x + a)
        x = self.norm2(x + self.ffn(x))
        return x.mean(1), w


class LNP_MFGO_Adversarial(nn.Module):
    """
    Full LNP-MFGO with:
    - Multi-component GIN encoder
    - Cross-component attention
    - Descriptor branch (Tier 1 features)
    - Multi-task prediction heads (PS, PDI, EE, Zeta, Bioactivity)
    - Adversarial study discriminator with gradient reversal
    """
    def __init__(self, atom_dim=ATOM_DIM, hidden=128, n_gin=3,
                 tab_gnn_dim=8, desc_dim=475, n_studies=60, drop=0.2):
        super().__init__()
        
        # --- GNN branch ---
        self.gins = nn.ModuleDict({
            n: GINEncoder(atom_dim, hidden, hidden, n_gin, drop) for n in LIPID_NAMES})
        self.cross_attn = CrossComponentAttention(hidden, 4, drop)
        self.tab_enc = nn.Sequential(nn.Linear(tab_gnn_dim, 32), nn.ReLU(), nn.Dropout(drop), nn.Linear(32, 32))
        
        # --- Descriptor branch ---
        self.desc_enc = nn.Sequential(
            nn.Linear(desc_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, 64),
        )
        
        # --- Fusion ---
        # GNN: hidden(128) + tab(32) = 160
        # Desc: 64
        # Ensemble gate learns to weight GNN vs Descriptor
        fusion_dim = 160 + 64  # = 224
        self.gate = nn.Sequential(nn.Linear(fusion_dim, 1), nn.Sigmoid())  # learned ensemble weight
        
        # --- Multi-task prediction heads ---
        self.heads = nn.ModuleDict({
            'ps': nn.Sequential(nn.Linear(fusion_dim, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, 1)),
            'pdi': nn.Sequential(nn.Linear(fusion_dim, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, 1)),
            'ee': nn.Sequential(nn.Linear(fusion_dim, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, 1)),
            'zeta': nn.Sequential(nn.Linear(fusion_dim, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, 1)),
            'bio': nn.Sequential(nn.Linear(fusion_dim, 64), nn.ReLU(), nn.Dropout(drop), nn.Linear(64, 4)),  # 4-class
        })
        
        # --- Adversarial study discriminator ---
        self.grl = GradientReversal(alpha=1.0)
        self.study_disc = nn.Sequential(
            nn.Linear(fusion_dim, 128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, n_studies),
        )
    
    def forward(self, graph_batch, tab_gnn, desc_features, grl_alpha=1.0):
        # GNN branch
        lipid_embs = []
        for n in LIPID_NAMES:
            x, ei, bna = graph_batch[n]
            lipid_embs.append(self.gins[n](x, ei, bna))
        form_emb, attn_w = self.cross_attn(torch.stack(lipid_embs, dim=1))
        tab_emb = self.tab_enc(tab_gnn)
        gnn_emb = torch.cat([form_emb, tab_emb], dim=-1)  # [B, 160]
        
        # Descriptor branch
        desc_emb = self.desc_enc(desc_features)  # [B, 64]
        
        # Fusion
        fused = torch.cat([gnn_emb, desc_emb], dim=-1)  # [B, 224]
        
        # Multi-task predictions
        preds = {}
        for task, head in self.heads.items():
            preds[task] = head(fused).squeeze(-1) if task != 'bio' else head(fused)
        
        # Adversarial study prediction
        self.grl.alpha = grl_alpha
        reversed_emb = self.grl(fused)
        study_logits = self.study_disc(reversed_emb)
        
        return preds, study_logits, attn_w


# Count parameters
_test = LNP_MFGO_Adversarial(desc_dim=len(TIER1), n_studies=N_STUDIES)
n_params = sum(p.numel() for p in _test.parameters())
print(f'LNP-MFGO-Adversarial: {n_params:,} parameters')
del _test
print('Adversarial model defined ✅')

---
## Section 3: Multi-Task Adversarial Training Loop

The training loop handles several complexities:

- **Missing targets:** Not all samples have all targets. The loss is computed only  
  over non-NaN samples per task, weighted equally.
- **GRL schedule:** λ ramps from 0→1 over training via the Ganin schedule.
- **Bioactivity as classification:** The bioactivity head outputs 4-class logits  
  (none/low/moderate/high), trained with cross-entropy loss.
- **Uncertainty weighting:** Each task loss is weighted by a learnable log-variance  
  parameter (Kendall et al., 2018), allowing the model to automatically balance tasks.

In [ ]:
# ============================================================
# 3.1 — Dataset & Collation
# ============================================================

class LNPMultiTaskDataset(Dataset):
    def __init__(self, indices, graphs, tab_gnn, desc_feats,
                 targets_dict, study_ids):
        self.indices = indices
        self.graphs = graphs
        self.tab_gnn = tab_gnn
        self.desc = desc_feats
        self.targets = targets_dict  # {'ps': array, 'pdi': array, ...}
        self.study_ids = study_ids
    
    def __len__(self): return len(self.indices)
    
    def __getitem__(self, idx):
        i = self.indices[idx]  # original df index (for graphs only)
        gs = {n: self.graphs[n][i] for n in LIPID_NAMES}
        tgts = {k: self.targets[k][idx] for k in self.targets}
        return gs, self.tab_gnn[idx], self.desc[idx], tgts, self.study_ids[idx]


def collate_mt(batch):
    gs_list, tab_list, desc_list, tgt_list, sid_list = zip(*batch)
    
    graph_batch = {}
    for n in LIPID_NAMES:
        xs, eis, bnas = [], [], []
        off = 0
        for sg in gs_list:
            g = sg[n]; xs.append(g['x']); eis.append(g['edge_index']+off); bnas.append(g['num_atoms']); off += g['num_atoms']
        graph_batch[n] = (torch.cat(xs), torch.cat(eis, 1), bnas)
    
    tab = torch.FloatTensor(np.array(tab_list))
    desc = torch.FloatTensor(np.array(desc_list))
    targets = {}
    for k in tgt_list[0]:
        targets[k] = torch.FloatTensor([t[k] for t in tgt_list])
    sids = torch.LongTensor(np.array(sid_list))
    return graph_batch, tab, desc, targets, sids

print('Multi-task dataset ready ✅')

In [ ]:
# ============================================================
# 3.2 — Training Function with Adversarial Loss
# ============================================================

def train_adversarial(model, train_loader, val_loader, device,
                       epochs=150, lr=5e-4, patience=30, adv_weight=0.1):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    mse = nn.MSELoss()
    ce = nn.CrossEntropyLoss()
    
    # Learnable task weights (uncertainty weighting)
    log_vars = nn.ParameterDict({
        t: nn.Parameter(torch.zeros(1, device=device)) for t in ['ps','pdi','ee','zeta','bio']
    }).to(device)
    optimizer.add_param_group({'params': log_vars.parameters()})
    
    best_val = float('inf'); best_state = None; pat = 0
    
    for epoch in range(epochs):
        grl_alpha = grl_lambda_schedule(epoch, epochs) * adv_weight
        
        # --- Train ---
        model.train(); total_loss = 0; nb = 0
        for gb, tab, desc, tgts, sids in train_loader:
            # Move to device
            gb_d = {n: (gb[n][0].to(device), gb[n][1].to(device), gb[n][2]) for n in LIPID_NAMES}
            tab_d = tab.to(device); desc_d = desc.to(device); sids_d = sids.to(device)
            tgts_d = {k: v.to(device) for k, v in tgts.items()}
            
            optimizer.zero_grad()
            preds, study_logits, _ = model(gb_d, tab_d, desc_d, grl_alpha)
            
            # Multi-task loss with uncertainty weighting
            loss = torch.tensor(0.0, device=device)
            n_tasks = 0
            for task in ['ps', 'pdi', 'ee', 'zeta']:
                mask = ~torch.isnan(tgts_d[task])
                if mask.sum() > 0:
                    task_loss = mse(preds[task][mask], tgts_d[task][mask])
                    # Uncertainty weighting: L_task / (2*σ²) + log(σ)
                    precision = torch.exp(-log_vars[task])
                    loss = loss + precision * task_loss + log_vars[task]
                    n_tasks += 1
            
            # Bioactivity (classification)
            bio_mask = ~torch.isnan(tgts_d['bio'])
            if bio_mask.sum() > 0:
                bio_tgt = tgts_d['bio'][bio_mask].long()
                bio_pred = preds['bio'][bio_mask]
                bio_loss = ce(bio_pred, bio_tgt)
                precision_bio = torch.exp(-log_vars['bio'])
                loss = loss + precision_bio * bio_loss + log_vars['bio']
                n_tasks += 1
            
            # Adversarial loss (study discrimination)
            adv_loss = ce(study_logits, sids_d)
            loss = loss + adv_weight * adv_loss  # GRL already reverses gradient
            
            if n_tasks > 0:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item(); nb += 1
        scheduler.step()
        
        # --- Validate (particle size only for early stopping) ---
        model.eval(); val_loss = 0; nv = 0
        with torch.no_grad():
            for gb, tab, desc, tgts, sids in val_loader:
                gb_d = {n: (gb[n][0].to(device), gb[n][1].to(device), gb[n][2]) for n in LIPID_NAMES}
                preds, _, _ = model(gb_d, tab.to(device), desc.to(device), 0.0)
                mask = ~torch.isnan(tgts['ps'].to(device))
                if mask.sum() > 0:
                    val_loss += mse(preds['ps'][mask], tgts['ps'].to(device)[mask]).item()
                    nv += 1
        val_loss /= max(nv, 1)
        
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience: break
    
    model.load_state_dict(best_state)
    return model, epoch + 1

print('Adversarial training loop ready ✅')

---
## Section 4: LOSO-CV — The Full Benchmark

We run the adversarial multi-task model on the same LOSO-CV protocol  
as Phases 2A and 2B, enabling direct comparison across all approaches.

In [ ]:
# ============================================================
# 4.1 — Prepare Data Arrays
# ============================================================

# Target arrays (NaN where missing)
target_arrays = {}
for tname in ['ps', 'pdi', 'ee', 'zeta']:
    col = f'{tname}_znorm'
    target_arrays[tname] = df[col].values.astype(np.float32)

target_arrays['bio'] = df['bio_ordinal'].values.astype(np.float32)  # NaN where missing

# Descriptor features
desc_data = df[TIER1].fillna(df[TIER1].median()).values.astype(np.float32)
# GNN tabular features
tab_gnn_data = df[TABULAR_GNN].fillna(df[TABULAR_GNN].median()).values.astype(np.float32)
# Study IDs
study_ids_arr = df['study_id'].values
studies_series = df['paper_doi']

# Usable studies
study_counts = studies_series.value_counts()
usable_studies = study_counts[study_counts >= 5].index.tolist()

print(f'Samples: {len(df)}')
print(f'Usable LOSO studies: {len(usable_studies)}')
for t in ['ps','pdi','ee','zeta','bio']:
    n_valid = np.isfinite(target_arrays[t]).sum()
    print(f'  {t}: {n_valid} non-NaN')

In [ ]:
# ============================================================
# 4.2 — Run LOSO-CV
# ============================================================
# Expected runtime: 45-120 min on GPU.

BATCH_SIZE = 32
EPOCHS = 150
PATIENCE = 30
LR = 5e-4
ADV_WEIGHT = 0.1

adv_fold_results = []
adv_all_attn = []
t_start = time.time()

def eval_metrics(yt, yp):
    m = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[m], yp[m]
    if len(yt) < 3:
        return {'R2': np.nan, 'RMSE': np.nan, 'MAE': np.nan,
                'Spearman_rho': np.nan, 'N': len(yt)}
    return {'R2': r2_score(yt, yp), 'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'MAE': mean_absolute_error(yt, yp), 'Spearman_rho': spearmanr(yt, yp)[0],
            'N': len(yt)}

for fold_i, test_study in enumerate(usable_studies):
    test_mask = (studies_series.values == test_study)
    tv_idx = np.where(~test_mask)[0]
    test_idx = np.where(test_mask)[0]
    if len(test_idx) < 3 or len(tv_idx) < 20: continue
    
    np.random.seed(SEED + fold_i)
    vs = max(5, int(0.1 * len(tv_idx)))
    perm = np.random.permutation(len(tv_idx))
    val_idx = tv_idx[perm[:vs]]
    train_idx = tv_idx[perm[vs:]]
    
    # Scale descriptor features
    dsc = StandardScaler()
    d_tr = dsc.fit_transform(desc_data[train_idx])
    d_va = dsc.transform(desc_data[val_idx])
    d_te = dsc.transform(desc_data[test_idx])
    
    tsc = StandardScaler()
    t_tr = tsc.fit_transform(tab_gnn_data[train_idx])
    t_va = tsc.transform(tab_gnn_data[val_idx])
    t_te = tsc.transform(tab_gnn_data[test_idx])
    
    # Datasets
    tgt_tr = {k: target_arrays[k][train_idx] for k in target_arrays}
    tgt_va = {k: target_arrays[k][val_idx] for k in target_arrays}
    tgt_te = {k: target_arrays[k][test_idx] for k in target_arrays}
    
    train_ds = LNPMultiTaskDataset(train_idx, all_graphs, t_tr, d_tr, tgt_tr, study_ids_arr[train_idx])
    val_ds = LNPMultiTaskDataset(val_idx, all_graphs, t_va, d_va, tgt_va, study_ids_arr[val_idx])
    test_ds = LNPMultiTaskDataset(test_idx, all_graphs, t_te, d_te, tgt_te, study_ids_arr[test_idx])
    
    train_ld = DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=collate_mt, drop_last=False)
    val_ld = DataLoader(val_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_mt)
    test_ld = DataLoader(test_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_mt)
    
    # Train
    model = LNP_MFGO_Adversarial(
        desc_dim=d_tr.shape[1], n_studies=int(N_STUDIES), drop=0.2
    ).to(DEVICE)
    model, n_ep = train_adversarial(model, train_ld, val_ld, DEVICE,
                                     EPOCHS, LR, PATIENCE, ADV_WEIGHT)
    
    # Evaluate on test
    model.eval()
    all_preds = {t: [] for t in ['ps','pdi','ee','zeta','bio']}
    all_trues = {t: [] for t in ['ps','pdi','ee','zeta','bio']}
    all_attn = []
    with torch.no_grad():
        for gb, tab, desc, tgts, sids in test_ld:
            gb_d = {n: (gb[n][0].to(DEVICE), gb[n][1].to(DEVICE), gb[n][2]) for n in LIPID_NAMES}
            preds, _, attn_w = model(gb_d, tab.to(DEVICE), desc.to(DEVICE), 0.0)
            for t in ['ps','pdi','ee','zeta']:
                all_preds[t].append(preds[t].cpu().numpy())
                all_trues[t].append(tgts[t].numpy())
            all_preds['bio'].append(preds['bio'].cpu().numpy())
            all_trues['bio'].append(tgts['bio'].numpy())
            all_attn.append(attn_w.cpu().numpy())
    
    # Metrics per task
    row = {'study': test_study, 'n_epochs': n_ep}
    for t in ['ps','pdi','ee','zeta']:
        yp = np.concatenate(all_preds[t])
        yt = np.concatenate(all_trues[t])
        m = eval_metrics(yt, yp)
        for k, v in m.items(): row[f'{t}_{k}'] = v
    
    # Bio accuracy
    bio_p = np.concatenate(all_preds['bio'])
    bio_t = np.concatenate(all_trues['bio'])
    bio_mask = np.isfinite(bio_t)
    if bio_mask.sum() > 0:
        bio_pred_cls = bio_p[bio_mask].argmax(axis=1)
        bio_true_cls = bio_t[bio_mask].astype(int)
        row['bio_accuracy'] = (bio_pred_cls == bio_true_cls).mean()
        row['bio_N'] = int(bio_mask.sum())
    
    adv_fold_results.append(row)
    adv_all_attn.append(np.concatenate(all_attn))
    
    elapsed = time.time() - t_start
    eta = elapsed / (fold_i+1) * (len(usable_studies)-fold_i-1)
    ps_r2 = row.get('ps_R2', np.nan)
    ps_r2_str = f'{ps_r2:+.4f}' if np.isfinite(ps_r2) else 'N/A'
    print(f'  Fold {fold_i+1:2d}/{len(usable_studies)}  PS_R²={ps_r2_str}  '
          f'({n_ep}ep, ETA {eta/60:.0f}min)')
    
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'\n✅ Done: {len(adv_fold_results)} folds, {(time.time()-t_start)/60:.1f} min total')

---
## Section 5: Results — Adversarial Multi-Task vs All Prior Approaches

In [ ]:
# ============================================================
# 5.1 — Compile & Compare All Approaches
# ============================================================

adv_df = pd.DataFrame(adv_fold_results)

# Particle size summary
ps_r2 = adv_df['ps_R2'].dropna()
ps_rho = adv_df['ps_Spearman_rho'].dropna()

print('ADVERSARIAL MULTI-TASK MODEL — LOSO-CV RESULTS')
print('='*70)
print(f'  Particle Size:  R²={ps_r2.mean():+.4f}±{ps_r2.std():.4f}  '
      f'ρ={ps_rho.mean():+.4f}  %Pos={100*(ps_r2>0).mean():.1f}%')

for t in ['pdi', 'ee', 'zeta']:
    r2s = adv_df[f'{t}_R2'].dropna()
    rhos = adv_df[f'{t}_Spearman_rho'].dropna()
    if len(r2s) > 0:
        print(f'  {t.upper():15s}: R²={r2s.mean():+.4f}±{r2s.std():.4f}  '
              f'ρ={rhos.mean():+.4f}  %Pos={100*(r2s>0).mean():.1f}%')

bio_acc = adv_df['bio_accuracy'].dropna()
if len(bio_acc) > 0:
    print(f'  Bioactivity:    Acc={bio_acc.mean():.3f}±{bio_acc.std():.3f}  '
          f'(random=0.25 for 4-class)')

# Full comparison table
print(f'\n{"="*80}')
print('GRAND COMPARISON — Particle Size, Z-Normalised, LOSO-CV')
print(f'{"="*80}')
print(f'{"Model":35s} {"Mean R²":>10s} {"Spearman ρ":>12s} {"% Positive":>10s}')
print('-'*80)
print(f'  {">>> Adv-MT (Phase 2C)":33s} {ps_r2.mean():+10.4f} '
      f'{ps_rho.mean():+12.4f} {100*(ps_r2>0).mean():9.1f}%')
# Phase 2B GNN
p2b_file = OUTPUT_DIR / 'phase2b_gnn_fold_results.csv'
if p2b_file.exists():
    p2b = pd.read_csv(p2b_file)
    # Trimmed (remove catastrophic outliers)
    p2b_clean = p2b[p2b['R2'] > -10]
    print(f'  {"GNN only (Phase 2B, trimmed)":33s} {p2b_clean["R2"].mean():+10.4f} '
          f'{p2b_clean["Spearman_rho"].mean():+12.4f} {100*(p2b_clean["R2"]>0).mean():9.1f}%')
# Phase 2A baselines
baselines = [
    ('LightGBM (Phase 2A)', 0.0181, 0.124, 38.5),
    ('XGBoost (Phase 2A)', 0.0154, 0.180, 46.2),
    ('RF tuned (Phase 2A)', 0.0019, 0.178, 51.3),
]
for name, r2, rho, pct in baselines:
    print(f'  {name:33s} {r2:+10.4f} {rho:+12.4f} {pct:9.1f}%')

# Save
adv_df.to_csv(OUTPUT_DIR / 'phase2c_adversarial_fold_results.csv', index=False)
print(f'\nSaved: phase2c_adversarial_fold_results.csv')

In [ ]:
# ============================================================
# 5.2 — Multi-Task Results Figure
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: R² per task
ax = axes[0]
task_data = []
for t, label in [('ps','Particle Size'), ('pdi','PDI'), ('ee','Encaps. Eff.'), ('zeta','Zeta Pot.')]:
    r2s = adv_df[f'{t}_R2'].dropna()
    if len(r2s) > 0:
        task_data.append({'Task': label, 'Mean_R2': r2s.mean(), 'Std_R2': r2s.std(),
                          'Pct_Pos': 100*(r2s>0).mean()})

if task_data:
    tdf = pd.DataFrame(task_data).sort_values('Mean_R2')
    colors = [COLORS['tertiary'] if r > 0 else COLORS['secondary'] for r in tdf['Mean_R2']]
    ax.barh(range(len(tdf)), tdf['Mean_R2'], xerr=tdf['Std_R2'],
            color=colors, alpha=0.8, capsize=5, edgecolor='white')
    ax.set_yticks(range(len(tdf)))
    ax.set_yticklabels([f"{r['Task']}\n({r['Pct_Pos']:.0f}% pos)" for _, r in tdf.iterrows()])
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('Mean R² (LOSO-CV)')
    ax.set_title('Multi-Task Performance\n(Adversarial, Z-Normalised)', fontweight='bold')

# Right: Per-fold PS R²
ax = axes[1]
sorted_ps = adv_df.sort_values('ps_R2', ascending=False)
ps_vals = sorted_ps['ps_R2'].dropna()
colors_f = [COLORS['tertiary'] if r > 0 else COLORS['secondary'] for r in ps_vals]
ax.barh(range(len(ps_vals)), ps_vals.values, color=colors_f, alpha=0.8)
ax.set_yticks(range(len(ps_vals)))
ax.set_yticklabels([str(s).split('/')[-1][:18] for s in sorted_ps['study'].iloc[:len(ps_vals)]], fontsize=7)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Test R²')
ax.set_title(f'Particle Size R² per Study\n'
             f'Mean={ps_r2.mean():+.4f}, {100*(ps_r2>0).mean():.0f}% positive', fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'fig13_adversarial_results.png', bbox_inches='tight')
plt.show()
print('Saved: fig13_adversarial_results.png')

In [ ]:
# ============================================================
# 5.3 — Save All Phase 2C Outputs
# ============================================================

# Attention weights
if adv_all_attn:
    all_attn_np = np.concatenate(adv_all_attn)
    np.save(OUTPUT_DIR / 'phase2c_attention_weights.npy', all_attn_np)
    # Mean attention
    mean_attn = all_attn_np.mean(0)
    print('Mean attention (Adversarial):')
    for i, n in enumerate(LIPID_NAMES):
        print(f'  {n:12s}: {mean_attn[i]}')

print(f'\nAll Phase 2C outputs saved to {OUTPUT_DIR}/')
print('Files: phase2c_adversarial_fold_results.csv, fig13, attention weights')

---
## Summary & Next Steps

### What Phase 2C adds to the paper:
1. **Adversarial domain adaptation** — first application to LNP property prediction
2. **Multi-task learning** — joint prediction of 4 properties + bioactivity
3. **GNN + Descriptor ensemble** — combines complementary representations
4. **Learned task weighting** — automatic balancing via uncertainty weighting

### Files to upload for next phase:
- `phase2c_adversarial_fold_results.csv`

### Remaining phases:
- **Phase 3:** Conditional Formulation Generator (CVAE)
- **Phase 4:** Interpretability + final paper figures
- **Phase 5:** Manuscript assembly